# 모듈 00: 환경 설정

LangChain을 배운 여러분, 이제 한 단계 더 나아갑니다.

LangChain으로 AI 파이프라인을 만들어봤다면, 이제 **분기, 루프, 조건 판단**이 가능한 진짜 에이전트를 만들 차례입니다. LangGraph가 그 도구입니다.

---

## 이 노트북에서 할 것

| 단계 | 내용 |
|------|------|
| 1 | LangGraph가 무엇인지, LangChain과 어떤 관계인지 이해 |
| 2 | `langgraph` 패키지 설치 |
| 3 | 기존 `.env`에서 API 키 로드 확인 |
| 4 | LangGraph 동작 가능 여부 종합 검증 |

**예상 소요 시간**: 약 15분  
**전제 조건**: LangChain 모듈 00 완료 (UV, `.env`, API 키 설정)

## 학습 목표

이 모듈을 완료하면 다음 세 가지를 할 수 있습니다:

1. **LangGraph와 LangChain의 관계 설명**: LangChain이 해결하지 못하는 문제가 무엇인지, LangGraph가 어떻게 그 한계를 극복하는지 설명할 수 있습니다.

2. **`langgraph` 패키지 설치 및 버전 확인**: `uv pip install`로 LangGraph를 설치하고, 정상 설치 여부를 버전으로 확인할 수 있습니다.

3. **기존 `.env`에서 API 키를 로드해 LangGraph에서 사용할 준비**: LangChain 모듈 00에서 생성한 `.env` 파일을 LangGraph 프로젝트에서도 참조해 API 키를 불러올 수 있습니다.

## 전체 흐름도

이 노트북의 전체 흐름을 한눈에 보면 이렇습니다:

```
┌─────────────────────────────────────────────────┐
│                                                 │
│  [1] LangGraph 개념 이해 (LangChain과의 관계)   │
│       ↓                                         │
│  [2] langgraph 패키지 설치                      │
│       ↓                                         │
│  [3] 기존 .env API 키 확인                      │
│       ↓                                         │
│  [4] 종합 환경 검증                             │
│                                                 │
└─────────────────────────────────────────────────┘
```

> **팁**: 셀을 순서대로 실행하세요! 위에서 아래로 하나씩 `Shift + Enter`를 눌러 실행하면 됩니다.

---

## 섹션 2: LangGraph란?

### LangChain의 한계

LangChain은 `prompt → LLM → parser` 형태의 **직선 파이프라인**에 최적화되어 있습니다. 하지만 실제 에이전트를 만들다 보면 이런 상황이 생깁니다:

- **조건 분기**: "LLM이 '툴 사용 필요'라고 판단하면 툴로 가고, 아니면 바로 응답"
- **루프**: "결과가 충분하지 않으면 다시 검색"
- **되돌아가기**: "에러가 났을 때 이전 단계로 돌아가서 재시도"

LangChain 체인만으로 이런 흐름을 구현하면 코드가 복잡해지고 관리가 어려워집니다.

### LangGraph = 상태 기계 기반 그래프 프레임워크

LangGraph는 **노드(Node)**와 **엣지(Edge)**로 워크플로우를 정의합니다:

- **노드**: 실행할 작업 (LLM 호출, 툴 실행, 데이터 처리 등)
- **엣지**: 노드 사이의 연결 (조건부 분기 가능)
- **상태(State)**: 그래프 전체를 관통하는 공유 데이터

### 비유로 이해하기

- **LangChain** = 컨베이어 벨트: 물건이 한 방향으로만 흘러갑니다
- **LangGraph** = 교통 신호가 있는 도로망: 상황에 따라 다른 길로 갑니다

### LangChain vs LangGraph 비교

| 항목 | LangChain (체인) | LangGraph (그래프) |
|------|-----------------|-------------------|
| 흐름 | 직선 | 분기/루프 가능 |
| 구조 | prompt → LLM → parser | 노드 + 엣지 |
| 상태 관리 | 제한적 | 명시적 State 타입 |
| 적합한 경우 | 단순 파이프라인 | 에이전트, 조건 분기 |

### 중요: LangGraph는 LangChain 위에서 동작합니다

LangGraph는 LangChain을 **대체**하는 것이 아니라 **확장**합니다. 내부적으로 `langchain-core`에 의존하며, LangChain의 모든 컴포넌트(ChatModel, Tool, PromptTemplate 등)를 그대로 사용할 수 있습니다.

---

## 섹션 3: 패키지 설치

### 설치할 패키지들

LangGraph를 사용하려면 아래 패키지들이 필요합니다:

| 패키지 | 역할 |
|--------|------|
| `langgraph` | 그래프 기반 워크플로우 프레임워크 |
| `langchain-google-genai` | (이미 설치됨) 재설치로 최신 버전 확인 |
| `python-dotenv` | (이미 설치됨) `.env` 파일 로드 |

`langchain-core`는 `langgraph`의 의존성으로 자동 설치됩니다.

In [ ]:
# 필요한 패키지 설치
# uv pip install로 설치합니다 (pip보다 빠릅니다)
print('패키지 설치를 시작합니다...')
print()

!uv pip install langgraph langchain-google-genai python-dotenv -q

print()
print('설치 완료! 다음 셀로 넘어가서 버전을 확인하세요.')

In [ ]:
# 설치된 패키지 버전 확인
# 아래 출력이 모두 [OK]로 나오면 설치가 성공한 것입니다

print('=== 패키지 설치 확인 ===')
print()

try:
    import langgraph
    print(f'[OK] langgraph: {langgraph.__version__}')
except ImportError:
    print('[FAIL] langgraph: 설치 실패 - 위 셀을 다시 실행해보세요')

try:
    import langchain_core
    print(f'[OK] langchain-core: {langchain_core.__version__}')
except ImportError:
    print('[FAIL] langchain-core: 설치 실패')

try:
    import langchain_google_genai
    print(f'[OK] langchain-google-genai: {langchain_google_genai.__version__}')
except ImportError:
    print('[FAIL] langchain-google-genai: 설치 실패')

try:
    import dotenv
    print(f'[OK] python-dotenv: {dotenv.__version__}')
except ImportError:
    print('[FAIL] python-dotenv: 설치 실패')

print()
print('모두 [OK]라면 패키지 설치 완료!')

---

## 섹션 4: API 키 확인

### LangChain 모듈 00에서 이미 설정 완료

API 키 발급과 `.env` 파일 생성은 LangChain 모듈 00에서 이미 마쳤습니다. 여기서는 그 `.env` 파일을 LangGraph 프로젝트에서 **로드할 수 있는지만 확인**합니다.

### 폴더 구조 이해

LangChain과 LangGraph는 형제 폴더 관계입니다:

```
10X 생산성 에이전트 .../
├── LangChain/
│   ├── .env        ← API 키가 여기 있습니다
│   └── 00_setup/
└── LangGraph/
    └── 00_setup/   ← 지금 이 노트북 위치
```

이 노트북은 `LangGraph/00_setup/`에 있으므로, 두 단계 위로 올라가면 프로젝트 루트가 나옵니다. 거기서 `LangChain/.env`를 참조합니다.

> **주의**: `LangChain/.env`가 없으면 아래 셀에서 [FAIL]이 나옵니다. 그 경우 LangChain 모듈 00으로 돌아가서 `.env`를 먼저 생성하거나, `LangGraph/` 루트에 별도로 `.env`를 복사하세요.

In [ ]:
# .env 로드 및 API 키 확인
import os
from dotenv import load_dotenv

# 경로 계산
# 이 노트북: .../LangGraph/00_setup/__file__
notebook_dir = os.path.dirname(os.path.abspath('__file__'))   # .../LangGraph/00_setup
langgraph_root = os.path.dirname(notebook_dir)                 # .../LangGraph
project_root = os.path.dirname(langgraph_root)                 # .../10X 생산성...

# LangChain 폴더의 .env를 우선 참조
langchain_env_path = os.path.join(project_root, 'LangChain', '.env')
# 대안: LangGraph 루트의 .env
langgraph_env_path = os.path.join(langgraph_root, '.env')

print('=== .env 파일 탐색 ===')
print()

if os.path.exists(langchain_env_path):
    loaded = load_dotenv(langchain_env_path)
    print(f'[OK] LangChain/.env 발견: {langchain_env_path}')
elif os.path.exists(langgraph_env_path):
    loaded = load_dotenv(langgraph_env_path)
    print(f'[OK] LangGraph/.env 발견: {langgraph_env_path}')
else:
    loaded = False
    print('[FAIL] .env 파일을 찾을 수 없습니다.')
    print()
    print('[주의] 해결 방법:')
    print('  1. LangChain 모듈 00으로 돌아가서 .env를 생성하세요.')
    print(f'  2. 또는 아래 경로에 .env 파일을 직접 복사하세요:')
    print(f'     {langgraph_env_path}')

print()
print('=== API 키 확인 ===')
api_key = os.getenv('GOOGLE_API_KEY')
if api_key and len(api_key) > 10:
    masked = api_key[:10] + '...' + api_key[-4:]
    print(f'[OK] GOOGLE_API_KEY: {masked}')
    print('API 키가 정상적으로 로드되었습니다!')
else:
    print('[FAIL] GOOGLE_API_KEY가 없거나 설정되지 않았습니다.')
    print('LangChain 모듈 00에서 .env 파일을 생성했는지 확인하세요.')

---

## 섹션 5: 종합 검증

### 최종 점검 방법

모든 설정이 끝났습니다! 이제 마지막으로 전체 환경이 올바르게 설정되었는지 확인합니다.

아래 두 개의 셀을 실행하면:

1. **종합 체크** - 각 항목별로 [OK] 또는 [FAIL]로 상태를 보여줍니다
2. **StateGraph import 테스트** - LangGraph의 핵심 클래스를 불러올 수 있는지 확인합니다

모든 항목이 [OK]라면 **모듈 01로 넘어갈 준비가 완료**된 것입니다!

In [ ]:
# 종합 환경 체크
import os
import sys
from dotenv import load_dotenv

def check_all():
    results = []

    # 1. Python 버전 확인
    v = sys.version_info
    if v >= (3, 10):
        results.append(('Python 버전', True, f'{v.major}.{v.minor}.{v.micro}'))
    else:
        results.append(('Python 버전', False, f'{v.major}.{v.minor} (3.10 이상 권장)'))

    # 2. langgraph 확인
    try:
        import langgraph
        results.append(('langgraph', True, langgraph.__version__))
    except ImportError:
        results.append(('langgraph', False, '미설치'))

    # 3. langchain-core 확인
    try:
        import langchain_core
        results.append(('langchain-core', True, langchain_core.__version__))
    except ImportError:
        results.append(('langchain-core', False, '미설치'))

    # 4. langchain-google-genai 확인
    try:
        import langchain_google_genai
        results.append(('langchain-google-genai', True, langchain_google_genai.__version__))
    except ImportError:
        results.append(('langchain-google-genai', False, '미설치'))

    # 5. GOOGLE_API_KEY 확인
    notebook_dir = os.path.dirname(os.path.abspath('__file__'))
    langgraph_root = os.path.dirname(notebook_dir)
    project_root = os.path.dirname(langgraph_root)
    langchain_env = os.path.join(project_root, 'LangChain', '.env')
    langgraph_env = os.path.join(langgraph_root, '.env')

    if os.path.exists(langchain_env):
        load_dotenv(langchain_env)
    elif os.path.exists(langgraph_env):
        load_dotenv(langgraph_env)

    api_key = os.getenv('GOOGLE_API_KEY')
    if api_key and len(api_key) > 10:
        masked = api_key[:10] + '...'
        results.append(('GOOGLE_API_KEY', True, masked))
    else:
        results.append(('GOOGLE_API_KEY', False, '미설정'))

    # 결과 출력
    print('=' * 50)
    print('       환경 설정 종합 체크 결과')
    print('=' * 50)
    all_ok = True
    for name, ok, detail in results:
        status = '[OK]  ' if ok else '[FAIL]'
        print(f'{status} {name:<25} {detail}')
        if not ok:
            all_ok = False

    print('=' * 50)
    if all_ok:
        print('모든 항목 통과! 모듈 01로 넘어갈 준비가 됐습니다!')
    else:
        print('[FAIL] 항목을 확인하고 해당 셀을 다시 실행하세요.')

check_all()

In [ ]:
# LangGraph import 테스트
# 실제 그래프를 실행하지 않고, 핵심 클래스를 불러올 수 있는지만 확인합니다
print('=== LangGraph Import 테스트 ===')
print()

try:
    from langgraph.graph import StateGraph, END
    print('[OK] StateGraph import 성공!')
    print('[OK] END import 성공!')
    print()
    print('다음 모듈에서 이 클래스로 첫 번째 그래프를 만듭니다.')
except ImportError as e:
    print(f'[FAIL] Import 실패: {e}')
    print()
    print('아래 명령어로 패키지를 다시 설치해보세요:')
    print('  !uv pip install langgraph')

---

## 섹션 6: 마무리

### 배운 것 정리

모듈 00에서 배운 내용을 정리해봅시다:

#### 1. LangGraph 개념 (상태 기계, 그래프 구조)
- LangChain 체인은 직선 파이프라인 → 조건 분기, 루프가 어려움
- LangGraph는 노드 + 엣지로 워크플로우를 정의하는 그래프 프레임워크
- LangChain을 대체하는 것이 아니라 그 위에서 확장함

#### 2. `langgraph` 패키지 설치
- `!uv pip install langgraph` 명령어로 설치
- `langchain-core`는 의존성으로 자동 설치됨

#### 3. `.env` API 키 로드 패턴
- LangChain 모듈 00에서 생성한 `.env`를 형제 폴더에서 참조
- 경로: `project_root/LangChain/.env`

---

**핵심 코드 패턴 (앞으로 계속 사용!):**

```python
from langgraph.graph import StateGraph, END

# 그래프 생성 (상태 타입 지정)
graph = StateGraph(상태_타입)

# 노드 추가
graph.add_node('노드이름', 함수)

# 엣지 연결
graph.add_edge('노드이름', END)
```

## 다음 모듈 예고: 모듈 01 - 첫 번째 그래프

환경 설정이 완료되었습니다! 다음 모듈에서는 **StateGraph로 첫 번째 LangGraph 워크플로우**를 만들어봅니다.

### 모듈 01에서 배울 것

```python
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_google_genai import ChatGoogleGenerativeAI

# 상태 정의
class State(TypedDict):
    messages: list

# 그래프 생성
graph = StateGraph(State)

# 노드 추가 → 엣지 연결 → 컴파일 → 실행
app = graph.compile()
result = app.invoke({'messages': ['안녕하세요!']})
```

### 준비 체크리스트

다음 모듈로 넘어가기 전에 확인하세요:

- [ ] 종합 체크 셀에서 모든 항목이 [OK]인가요?
- [ ] `StateGraph` import가 성공했나요?
- [ ] `GOOGLE_API_KEY` 로드가 성공했나요?

모두 체크했다면 `01_first_graph/모듈01_첫번째_그래프.ipynb`를 열어보세요!

---

수고하셨습니다!